# 03 · Capa silver
Limpieza de `application_train` y agregación de cada tabla secundaria a una fila por cliente (`SK_ID_CURR`). Cada tabla se valida: filas = clientes distintos.

In [0]:
from pyspark.sql import functions as F

ESQ = "workspace.credit_risk"

# guardar escribe la tabla y comprueba que haya una fila por cliente:
def guardar(df, nombre):
    df.write.mode("overwrite").saveAsTable(f"{ESQ}.{nombre}")
    t = spark.table(f"{ESQ}.{nombre}")
    n, d = t.count(), t.select("SK_ID_CURR").distinct().count()
    print(f"{nombre}: {n:,} filas, {d:,} clientes -> {'OK' if n == d else 'REVISAR: hay duplicados'}")

In [0]:
app = spark.table(f"{ESQ}.bronze_application_train")

# application limpia, se aplican las decisiones de la exploración(centinela a NULL, años en vez de días):
app_s = (app
    .withColumn("DAYS_EMPLOYED", F.when(F.col("DAYS_EMPLOYED") == 365243, None).otherwise(F.col("DAYS_EMPLOYED")))
    .withColumn("EMPLOYED_YEARS", -F.col("DAYS_EMPLOYED") / 365.25)
    .withColumn("AGE_YEARS", -F.col("DAYS_BIRTH") / 365.25)
    .withColumn("CODE_GENDER", F.when(F.col("CODE_GENDER") == "XNA", None).otherwise(F.col("CODE_GENDER"))))

guardar(app_s, "silver_application")

In [0]:
# cuotas (installments), mide el comportamiento de pago pasado
inst = spark.table(f"{ESQ}.bronze_installments_payments")

inst_agg = (inst
    .withColumn("dias_atraso", F.col("DAYS_ENTRY_PAYMENT") - F.col("DAYS_INSTALMENT"))
    .withColumn("pago_faltante", F.col("AMT_INSTALMENT") - F.col("AMT_PAYMENT"))
    .groupBy("SK_ID_CURR")
    .agg(F.count("*").alias("INST_N_CUOTAS"),
         F.avg("dias_atraso").alias("INST_ATRASO_PROM"),
         F.max("dias_atraso").alias("INST_ATRASO_MAX"),
         F.avg((F.col("dias_atraso") > 0).cast("int")).alias("INST_PCT_TARDE"),
         F.avg((F.col("pago_faltante") > 0).cast("int")).alias("INST_PCT_PAGO_INCOMPLETO"),
         F.sum("pago_faltante").alias("INST_FALTANTE_TOTAL")))

guardar(inst_agg, "silver_inst_agg")

In [0]:
# buró (bureau). Resume los créditos en otras instituciones:
bur = spark.table(f"{ESQ}.bronze_bureau")

bur_agg = (bur.groupBy("SK_ID_CURR")
    .agg(F.count("*").alias("BUR_N_CREDITOS"),
         F.sum((F.col("CREDIT_ACTIVE") == "Active").cast("int")).alias("BUR_N_ACTIVOS"),
         F.sum((F.col("CREDIT_ACTIVE") == "Closed").cast("int")).alias("BUR_N_CERRADOS"),
         F.max("CREDIT_DAY_OVERDUE").alias("BUR_DIAS_MORA_MAX"),
         F.sum("AMT_CREDIT_SUM_OVERDUE").alias("BUR_MONTO_VENCIDO"),
         F.sum("AMT_CREDIT_SUM").alias("BUR_CREDITO_TOTAL"),
         F.sum("AMT_CREDIT_SUM_DEBT").alias("BUR_DEUDA_TOTAL"),
         F.avg("DAYS_CREDIT").alias("BUR_ANTIGUEDAD_PROM"))
    .withColumn("BUR_RATIO_DEUDA",
                F.when(F.col("BUR_CREDITO_TOTAL") > 0, F.col("BUR_DEUDA_TOTAL") / F.col("BUR_CREDITO_TOTAL"))))

guardar(bur_agg, "silver_bureau_agg")

In [0]:
# solicitudes previas (previous_application):
prev = spark.table(f"{ESQ}.bronze_previous_application")

prev_agg = (prev.groupBy("SK_ID_CURR")
    .agg(F.count("*").alias("PREV_N_SOLICITUDES"),
         F.sum((F.col("NAME_CONTRACT_STATUS") == "Approved").cast("int")).alias("PREV_N_APROBADAS"),
         F.sum((F.col("NAME_CONTRACT_STATUS") == "Refused").cast("int")).alias("PREV_N_RECHAZADAS"),
         F.avg("AMT_APPLICATION").alias("PREV_MONTO_SOLICITADO_PROM"),
         F.avg("AMT_CREDIT").alias("PREV_MONTO_OTORGADO_PROM"),
         F.max("DAYS_DECISION").alias("PREV_DIAS_ULTIMA_SOLICITUD"))
    .withColumn("PREV_PCT_RECHAZO", F.col("PREV_N_RECHAZADAS") / F.col("PREV_N_SOLICITUDES")))

guardar(prev_agg, "silver_prev_agg")

In [0]:
# tarjetas y POS (volumen moderado):
cc = spark.table(f"{ESQ}.bronze_credit_card_balance")
cc_agg = (cc
    .withColumn("uso", F.when(F.col("AMT_CREDIT_LIMIT_ACTUAL") > 0, F.col("AMT_BALANCE") / F.col("AMT_CREDIT_LIMIT_ACTUAL")))
    .groupBy("SK_ID_CURR")
    .agg(F.avg("uso").alias("CC_USO_PROM"),
         F.max("SK_DPD").alias("CC_DPD_MAX"),
         F.avg("AMT_BALANCE").alias("CC_SALDO_PROM"),
         F.countDistinct("SK_ID_PREV").alias("CC_N_TARJETAS")))
guardar(cc_agg, "silver_cc_agg")

pos = spark.table(f"{ESQ}.bronze_pos_cash_balance")
pos_agg = (pos.groupBy("SK_ID_CURR")
    .agg(F.max("SK_DPD").alias("POS_DPD_MAX"),
         F.avg((F.col("SK_DPD") > 0).cast("int")).alias("POS_PCT_MESES_MORA"),
         F.count("*").alias("POS_N_REGISTROS")))
guardar(pos_agg, "silver_pos_agg")

In [0]:
# bureau_balance (27 millones de filas). Se resume por crédito del buró y luego por cliente:
bb = spark.table(f"{ESQ}.bronze_bureau_balance")
bb_bur = (bb.groupBy("SK_ID_BUREAU")
    .agg(F.count("*").alias("meses"),
         F.avg(F.col("STATUS").isin("1", "2", "3", "4", "5").cast("int")).alias("pct_meses_mora")))

bb_agg = (spark.table(f"{ESQ}.bronze_bureau").select("SK_ID_CURR", "SK_ID_BUREAU")
    .join(bb_bur, "SK_ID_BUREAU", "left")
    .groupBy("SK_ID_CURR")
    .agg(F.avg("pct_meses_mora").alias("BB_PCT_MORA_PROM"),
         F.max("pct_meses_mora").alias("BB_PCT_MORA_MAX"),
         F.sum("meses").alias("BB_MESES_TOTAL")))
guardar(bb_agg, "silver_bb_agg")